# 06 — Figures

**Reads:** `data/processed/nowcasts.csv`, `results/tables/test_metrics.csv`, `shap_values.csv`,
`cv_choice.json`.

**Writes:** `results/figures/fig1_nowcasts.html`, `fig2_errors.html`, `fig3_shap.html`, and copies them
into the post folder when the website repository sits next to this one.

Figures carry a title, axis labels with units, a legend and a short source line; the explanation goes in
the post. Hover values are rounded to two decimals.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for d in (PROC, TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

import shutil
import plotly.graph_objects as go
from plotly_style import SITE, COLORS, FONT, base_layout, axis_style, source_note, save_html, save_html_controls
from nowcast_setup import FEATURE_LABELS, MODEL_LABELS

now = pd.read_csv(PROC / "nowcasts.csv", parse_dates=["release_date"])
now["date"] = pd.PeriodIndex(now["period"], freq="M").to_timestamp()
metrics = pd.read_csv(TAB / "test_metrics.csv", index_col="method")
sv = pd.read_csv(TAB / "shap_values.csv")
choice = json.loads((TAB / "cv_choice.json").read_text())

SOURCE = "BLS first prints and Reuters poll via LSEG;<br>Manheim (Cox Automotive); jet fuel via FRED"
MODEL_SOURCE = "author's calculations;<br>data from LSEG, Cox Automotive<br>and FRED."
SHORT = {   # short input labels so the bar charts stay readable on a phone
    "core_last": "Core: last print", "core_avg3": "Core: 3-print average", "core_avg12": "Core: 12-print average",
    "manheim_0": "Used cars: same month", "manheim_1": "Used cars: 1 month earlier",
    "manheim_2": "Used cars: 2 months earlier", "jet_0": "Jet fuel: same month", "jet_1": "Jet fuel: 1 month earlier",
}
POST = ROOT.parent / "Personal Website" / "blog" / "posts" / "post10"
LINE = {   # method -> (label, colour, dash, shown at first)
    "y": ("First print", COLORS["ink"], "solid", True),
    "poll": ("Reuters poll", COLORS["band"], "dot", True),
    "ols": ("Linear regression", COLORS["pre"], "solid", True),
    "average": ("Tree models", COLORS["post"], "solid", True),
    "avg12": ("12-print average", COLORS["grey"], "dashdot", "legendonly"),
    "last_print": ("Last print", COLORS["grey"], "dot", "legendonly"),
}

## Figure 1: first prints and nowcasts, month by month

In [2]:
fig = go.Figure()
for col, (label, color, dash, visible) in LINE.items():
    fig.add_trace(go.Scatter(
        x=now["date"], y=now[col], name=label, visible=visible,
        mode="lines+markers" if col == "y" else "lines",
        line=dict(color=color, dash=dash, width=2.4 if col == "y" else 1.8),
        marker=dict(size=5, color=color),
        hovertemplate=f"{label}<br>%{{x|%b %Y}}: %{{y:.2f}}%<extra></extra>"))
fig.update_layout(**base_layout("Core CPI: first print and nowcasts", height=560))
fig.update_layout(legend=dict(orientation="h", x=0, xanchor="left", y=1.02, yanchor="bottom",
                              entrywidth=0.5, entrywidthmode="fraction"),   # two entries per row on any screen
                  margin=dict(t=150, b=110))
fig.update_xaxes(**axis_style(None, dtick="M12", tickformat="%Y"))
fig.update_yaxes(**axis_style("% change from the previous month", ticksuffix="%"))
source_note(fig, SOURCE + ".", y=-0.12)
save_html(fig, FIG / "fig1_nowcasts.html")
fig.show()

## Figure 2: error by method, with a control for the period

In [3]:
ORDER = ["Last month's print (persistence)", "Average of the last 12 prints", "Reuters poll (median)",
         "Linear regression", "Random forest", "Gradient boosting (XGBoost)", "Average of random forest and XGBoost"]
BAR_LABEL = dict(zip(ORDER, ["Last print (persistence)", "12-print average", "Reuters poll", "Linear regression",
                             "Random forest", "XGBoost", "Tree models (average)"]))
PERIODS = {"All test months (Jan 2021 – Oct 2026)": "RMSE, all test months",
           "2021": "RMSE, 2021",
           "2022 – Oct 2026": "RMSE, 2022-2026"}
BAR = {m: (COLORS["grey"] if i < 2 else COLORS["band"] if i == 2 else COLORS["pre"] if i < 6 else COLORS["post"])
       for i, m in enumerate(ORDER)}
fig = go.Figure()
for p_label, col in PERIODS.items():
    vals = metrics.loc[ORDER, col]
    fig.add_trace(go.Bar(
        y=[BAR_LABEL[m] for m in ORDER], x=vals, orientation="h", name=p_label, marker_color=[BAR[m] for m in ORDER],
        text=[f"{v:.2f}" for v in vals], textposition="outside", cliponaxis=False,
        hovertemplate="%{y}<br>RMSE %{x:.2f} percentage points<extra></extra>", showlegend=False))
xmax = float(metrics[list(PERIODS.values())].max().max()) * 1.18
fig.update_layout(**base_layout("Nowcast error by method", height=530, legend_top=False))
fig.update_layout(margin=dict(l=10, t=70, b=150))
fig.update_xaxes(**axis_style("RMSE (pct. points)", range=[0, xmax], nticks=5))
fig.update_yaxes(**axis_style(None, autorange="reversed", showgrid=False, automargin=True))
source_note(fig, MODEL_SOURCE, y=-0.28)
keys = list(PERIODS)
save_html_controls(fig, FIG / "fig2_errors.html",
                   controls=[("period", "Test months:", keys)],
                   visibility={k: [k == j for j in keys] for k in keys})
fig.show()

## Figure 3: SHAP, two controls (model × view)

In [4]:
inputs = sv[sv["input"].isin(FEATURE_LABELS)].copy()
MODELS = {"Linear regression": "ols", "Random forest": "rf", "XGBoost": "xgb"}
VIEWS = {"Average size, all test months": "avg", "April 2021 nowcast": "2021-04"}
fig = go.Figure()
keys, ranges = [], []
for m_label, kind in MODELS.items():
    sub = inputs[inputs["model"] == kind]
    for v_label, view in VIEWS.items():
        if view == "avg":
            s = sub.assign(a=sub["shap"].abs()).groupby("input")["a"].mean()
            hover = "%{y}<br>average size %{x:.2f} percentage points<extra></extra>"
        else:
            s = sub[sub["period"] == view].set_index("input")["shap"]
            hover = "%{y}<br>contribution %{x:+.2f} percentage points<extra></extra>"
        s = s.reindex([c for c in FEATURE_LABELS if c in s.index])
        colors = [COLORS["post"] if v >= 0 else COLORS["pre"] for v in s.values]
        span = max(float(s.max()), 0) - min(float(s.min()), 0)
        ranges.append([min(float(s.min()), 0) - 0.3 * span * (s.min() < 0), max(float(s.max()), 0) + 0.25 * span])
        fig.add_trace(go.Bar(y=[SHORT[c] for c in s.index], x=s.values, orientation="h",
                             marker_color=colors,
                             text=["≈0" if abs(v) < 0.005 else (f"{v:+.2f}" if view != "avg" else f"{v:.2f}") for v in s.values],
                             textposition="outside", cliponaxis=False, hovertemplate=hover, showlegend=False))
        keys.append(f"{m_label}|{v_label}")
n = len(keys)
visibility = {k: [i == j for j in range(n)] for i, k in enumerate(keys)}
layouts = {k: {"xaxis.title.text": ("Average size (pct. points)"
                                    if k.endswith("all test months") else
                                    "April 2021 (pct. points)"),
               "xaxis.range": r} for k, r in zip(keys, ranges)}
fig.update_layout(**base_layout("What moved each nowcast: SHAP values", height=530, legend_top=False))
fig.update_layout(margin=dict(l=10, t=70, r=40, b=150))
fig.update_xaxes(**axis_style("Average size (pct. points)", range=ranges[0], nticks=5,
                              zeroline=True, zerolinecolor=SITE["axis"]))
fig.update_yaxes(**axis_style(None, autorange="reversed", showgrid=False, automargin=True))
source_note(fig, MODEL_SOURCE, y=-0.28)
save_html_controls(fig, FIG / "fig3_shap.html",
                   controls=[("model", "Model:", list(MODELS)), ("view", "Show:", list(VIEWS))],
                   visibility=visibility, layouts=layouts)
fig.show()

In [5]:
if POST.exists():
    for f in ("fig1_nowcasts.html", "fig2_errors.html", "fig3_shap.html"):
        shutil.copy(FIG / f, POST / f)
    print("copied to", POST)
else:
    print("post folder not found; copy results/figures/*.html into the post folder by hand")

post folder not found; copy results/figures/*.html into the post folder by hand
